# AlphaBuilder v4.0 - Stage 5 Training (`Phys-UViT 3D`, Kaggle `2 x Tesla T4`)

Notebook oficial da **Etapa 5** para treinamento supervisionado da arquitetura **`Phys-UViT 3D`** (`~4,9M` params, `0% padding em Z`) via **DDP (`torchrun --standalone --nproc_per_node=2`)** sobre `warmup_v3_3_kaggle.db` e `split_v3_3.json`:

- **`D4 x Z2 = 16x` simetrias 3D** + *negative sampling* com calibração física analítica de `log10(C)` (`C(3F) = 9 C(F)`)
- **Global Physics Transformer (`3D RoPE` + `AdaLN-Zero` + `Multi-Load Cross-Attn` + `SwiGLU`)** no *bottleneck* `16 x 8 x 2 = 256` tokens
- **Value Head Fatorada**: supervisiona `log10(Compliance)` (`lambda=0.20`) e `Conectividade` (`lambda=0.25`) em conjunto com `V(s)`, além de `Policy KL` (`1.0`) e `Blueprint` (`0.50`)
- Sem dependência de MONAI!


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

print("=" * 64)
print("AlphaBuilder v4.0 - Phys-UViT 3D Kaggle Training Pipeline")
print("=" * 64)
print(f"CUDA available: {torch.cuda.is_available()} | GPU count: {torch.cuda.device_count()}")
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f"  GPU {idx}: {torch.cuda.get_device_name(idx)} ({props.total_memory / 1024**3:.1f} GB)")


In [ ]:
# === Configuracao da Execucao (Stage 5 - Phys-UViT 3D) ===
PRESET = "kaggle_t4_dual_phys_uvit"
EPOCHS_OVERRIDE = 30        # Troque para 2 no smoke test de VRAM/velocidade; None usa default (30)
LAMBDA_BP_OVERRIDE = None
RESUME_CHECKPOINT = None

REPO_URL = "https://github.com/gustavomello9600/alphabuild.git"
REPO_DIR = Path("/tmp/alphabuild")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Active commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
# === Descoberta Automatica de warmup_v3_3_kaggle.db e split_v3_3.json ===
import shutil
import time

search_roots = [Path("/kaggle/input"), REPO_DIR / "data" / "compiled", Path("/kaggle/working")]

db_candidates = []
split_candidates = []
for root in search_roots:
    if root.exists():
        db_candidates.extend(sorted(root.rglob("warmup_v3_3_kaggle.db")))
        split_candidates.extend(sorted(root.rglob("split_v3_3.json")))

if not db_candidates:
    raise FileNotFoundError(
        "warmup_v3_3_kaggle.db nao encontrado em /kaggle/input! "
        "Anexe o Dataset Kaggle contendo warmup_v3_3_kaggle.db e split_v3_3.json ao notebook."
    )

SRC_DB_PATH = db_candidates[0]
LOCAL_DB_PATH = Path("/tmp/warmup_v3_3_kaggle.db")
if not LOCAL_DB_PATH.exists() or LOCAL_DB_PATH.stat().st_size != SRC_DB_PATH.stat().st_size:
    t0 = time.perf_counter()
    print(f"Copying {SRC_DB_PATH} -> {LOCAL_DB_PATH} (local SSD)...")
    shutil.copyfile(SRC_DB_PATH, LOCAL_DB_PATH)
    print(f"Copied in {time.perf_counter() - t0:.1f}s")

DB_PATH = LOCAL_DB_PATH
SPLIT_PATH = split_candidates[0] if split_candidates else None

print(f"Database: {DB_PATH} ({DB_PATH.stat().st_size / 1e6:.2f} MB)")
print(f"Split:    {SPLIT_PATH}")


In [ ]:
# === Treinamento Multi-GPU (DDP via torchrun) ou Single-GPU ===
gpu_count = torch.cuda.device_count()
if gpu_count <= 1 and "dual" in PRESET:
    PRESET = PRESET.replace("_dual_", "_")
    print(f"Single GPU detectada -> ajustando preset para {PRESET}")

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
LOG_DIR = Path("/kaggle/working/logs")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

if gpu_count > 1:
    cmd = [
        "torchrun",
        "--standalone",
        f"--nproc_per_node={gpu_count}",
        "alphabuilder/src/neural/train_pipeline.py",
    ]
else:
    cmd = [sys.executable, "alphabuilder/src/neural/train_pipeline.py"]

cmd.extend([
    "--config", PRESET,
    "--db-path", str(DB_PATH),
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--log-dir", str(LOG_DIR),
])
if SPLIT_PATH is not None:
    cmd.extend(["--split-json", str(SPLIT_PATH)])
if EPOCHS_OVERRIDE is not None:
    cmd.extend(["--epochs", str(EPOCHS_OVERRIDE)])
if LAMBDA_BP_OVERRIDE is not None:
    cmd.extend(["--lambda-bp", str(LAMBDA_BP_OVERRIDE)])
if RESUME_CHECKPOINT:
    cmd.extend(["--resume", str(RESUME_CHECKPOINT)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# === Resumo, Curvas Phys-UViT 3D e Exportacao para /kaggle/working ===
import json
import shutil

history_path = CHECKPOINT_DIR / "metrics_history.json"
if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))
    if history:
        best_ep = min(history, key=lambda h: h["val"].get("loss", float("inf")))
        print(f"Melhor epoca: {best_ep['epoch']} | Val Loss={best_ep['val']['loss']:.4f} | "
              f"Core={best_ep['val'].get('core_loss', best_ep['val']['loss']):.4f} | "
              f"Top1={best_ep['val'].get('policy_top1_acc', 0.0):.3f} | "
              f"MAE={best_ep['val'].get('value_mae', 0.0):.4f} | "
              f"Dice={best_ep['val'].get('blueprint_dice', 0.0):.3f} | "
              f"logC_MAE={best_ep['val'].get('log_c_mae', 0.0):.4f} | "
              f"ConnAcc={best_ep['val'].get('conn_acc', 0.0):.3f}")
    try:
        import matplotlib.pyplot as plt
        epochs = [h["epoch"] for h in history]
        fig, axes = plt.subplots(1, 4, figsize=(19, 4))
        axes[0].plot(epochs, [h["train"]["loss"] for h in history], label="Train Total")
        axes[0].plot(epochs, [h["val"]["loss"] for h in history], label="Val Total")
        axes[0].plot(epochs, [h["val"].get("core_loss", h["val"]["loss"]) for h in history], "--", label="Val Core v3.3")
        axes[0].set_title("Total & Core Loss")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, [h["val"].get("policy_top1_acc", 0.0) for h in history], label="Policy Top-1")
        axes[1].plot(epochs, [h["val"].get("blueprint_dice", 0.0) for h in history], label="Blueprint Dice")
        axes[1].set_title("Policy Top-1 & Blueprint Dice")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        axes[2].plot(epochs, [h["val"].get("value_mae", 0.0) for h in history], label="Value MAE")
        axes[2].plot(epochs, [h["val"].get("log_c_mae", 0.0) for h in history], label="log10(C) MAE")
        axes[2].set_title("Value & log10(C) MAE")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        axes[3].plot(epochs, [h["val"].get("conn_acc", 0.0) for h in history], label="Connectivity Acc")
        axes[3].set_title("Topological Connectivity Acc")
        axes[3].legend()
        axes[3].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("/kaggle/working/training_curves.png", dpi=150)
        plt.show()
    except Exception as exc:
        print("Plot skipped:", exc)

for fname in ("best_model.pt", "last_model.pt", "metrics_history.json"):
    src = CHECKPOINT_DIR / fname
    if src.exists():
        shutil.copy2(src, Path("/kaggle/working") / fname)
        print(f"Exported: /kaggle/working/{fname} ({src.stat().st_size / 1e6:.2f} MB)")
